In [1]:
import os, multiprocessing, psutil

print("CPU cores available:", multiprocessing.cpu_count())
print("Total RAM (GiB):", psutil.virtual_memory().total / (1024**3))
print("Current working directory:", os.getcwd())

CPU cores available: 128
Total RAM (GiB): 503.4748420715332
Current working directory: /home/rshayan1


In [2]:
# === 1. System info & Python version (sanity) ===
import sys, platform, subprocess, os, random, numpy as np
print("Python:", sys.version)
print("Platform:", platform.platform())
print("CWD:", os.getcwd())

Python: 3.12.9 | packaged by conda-forge | (main, Mar  4 2025, 22:48:41) [GCC 13.3.0]
Platform: Linux-4.18.0-553.77.1.el8_10.x86_64-x86_64-with-glibc2.28
CWD: /home/rshayan1


In [3]:
# VERIFY: imports/versions + thread envs
import sys, os, multiprocessing
def version_of(name, import_name=None):
    try:
        m = __import__(import_name or name)
        return getattr(m, "__version__", "unknown")
    except Exception as e:
        return f"NOT IMPORTABLE ({type(e).__name__}: {e})"

print("sys.executable:", sys.executable)
print("python:", sys.version.split()[0])
print("cpu_count:", multiprocessing.cpu_count())
print("Thread envs:", {k:os.environ.get(k) for k in ('OMP_NUM_THREADS','OPENBLAS_NUM_THREADS','MKL_NUM_THREADS')})
print()

pkgs = [
    ("numpy","numpy"),
    ("pandas","pandas"),
    ("scikit-learn","sklearn"),
    ("gplearn","gplearn"),
    ("xgboost","xgboost"),
    ("joblib","joblib"),
    ("pyarrow","pyarrow"),
]
for name, imp in pkgs:
    print(f"{name:12s} -> {version_of(name, imp)}")

sys.executable: /packages/apps/jupyter/2025-03-24/bin/python3.12
python: 3.12.9
cpu_count: 128
Thread envs: {'OMP_NUM_THREADS': None, 'OPENBLAS_NUM_THREADS': None, 'MKL_NUM_THREADS': None}

numpy        -> 1.26.4
pandas       -> 2.3.3
scikit-learn -> 1.2.2
gplearn      -> 0.4.2
xgboost      -> 2.1.1
joblib       -> 1.5.3
pyarrow      -> 22.0.0


In [4]:
# OPTIONAL: programmatic restart (the UI restart is usually safer)
import IPython
IPython.Application.instance().kernel.do_shutdown(True)

{'status': 'ok', 'restart': True}

In [1]:
# SET THREAD ENV for this kernel (run after restart)
import os, multiprocessing
CPU = 12   # <-- change this if you reserved a different number of cores on SOL
for k in ("OMP_NUM_THREADS","OPENBLAS_NUM_THREADS","MKL_NUM_THREADS","NUMEXPR_NUM_THREADS"):
    os.environ[k] = str(CPU)
print("Thread envs now:", {k:os.environ.get(k) for k in ("OMP_NUM_THREADS","OPENBLAS_NUM_THREADS","MKL_NUM_THREADS")})

Thread envs now: {'OMP_NUM_THREADS': '12', 'OPENBLAS_NUM_THREADS': '12', 'MKL_NUM_THREADS': '12'}


In [2]:
import numpy as np
import pandas as pd
import sklearn
import gplearn
import xgboost
import joblib

print("NumPy:", np.__version__)
print("Pandas:", pd.__version__)
print("sklearn:", sklearn.__version__)
print("gplearn:", gplearn.__version__)
print("xgboost:", xgboost.__version__)
print("joblib:", joblib.__version__)

NumPy: 1.26.4
Pandas: 2.3.3
sklearn: 1.2.2
gplearn: 0.4.2
xgboost: 2.1.1
joblib: 1.5.3


In [4]:
# === 2. Dataset loader (Strategy 2) ===
from sklearn.datasets import fetch_california_housing
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
import numpy as np, pandas as pd

def load_dataset(name="california"):
    """
    Returns: X_train, X_test, y_train, y_test, feature_names
    """
    if name.lower() in ("california","cal"):
        ds = fetch_california_housing(as_frame=True)
        X = ds.data
        y = ds.target
        feat_names = list(X.columns)
    else:
        raise ValueError("Currently supported: 'california'. (We can add 'energy', 'bike' next.)")

    X_train, X_test, y_train, y_test = train_test_split(
        X.values, y.values, test_size=0.2, random_state=42
    )
    # scale X (not y)
    scaler = StandardScaler().fit(X_train)
    X_train = scaler.transform(X_train)
    X_test  = scaler.transform(X_test)
    return X_train, X_test, y_train, y_test, feat_names

X_train, X_test, y_train, y_test, feat_names = load_dataset("california")
X_train.shape, X_test.shape, len(feat_names)

((16512, 8), (4128, 8), 8)

In [5]:
# === 3. Feature grouping into K groups (Strategy 2) ===
from sklearn.cluster import AgglomerativeClustering

def make_feature_groups(X, K):
    """
    X: (n_samples, n_features)
    Returns: list of lists of feature indices, length K
    """
    # correlation distance between features
    C = np.corrcoef(X, rowvar=False)
    D = 1.0 - np.abs(C)
    # Agglomerative expects condensed/metric; we use precomputed + average linkage
    # It ignores diagonal; ensure symmetry and zeros on diag
    np.fill_diagonal(D, 0.0)
    model = AgglomerativeClustering(
        metric="precomputed", linkage="average", n_clusters=K
    )
    labels = model.fit_predict(D)
    groups = []
    for g in range(K):
        groups.append(np.where(labels == g)[0].tolist())
    # sort smallest group first purely for readability
    groups = sorted(groups, key=len)
    return groups

K = 4  # you can change
groups = make_feature_groups(X_train, K)
print("K groups (by feature index):", groups)
print("Feature names by group:")
for i,g in enumerate(groups):
    print(f"  G{i+1}:", [feat_names[j] for j in g])

K groups (by feature index): [[5], [1, 4], [6, 7], [0, 2, 3]]
Feature names by group:
  G1: ['AveOccup']
  G2: ['HouseAge', 'Population']
  G3: ['Latitude', 'Longitude']
  G4: ['MedInc', 'AveRooms', 'AveBedrms']


In [6]:
# --- Ensure pandas-friendly shapes for the sweep ---
import numpy as np, pandas as pd

def ensure_df(X, feature_names=None):
    if isinstance(X, pd.DataFrame):
        return X
    if feature_names is None:
        # Try to use names from the California housing loader if available
        try:
            from sklearn.datasets import fetch_california_housing
            feature_names = fetch_california_housing(as_frame=True).feature_names
        except Exception:
            feature_names = [f"X{i}" for i in range(X.shape[1])]
    return pd.DataFrame(X, columns=feature_names)

def ensure_series(y, name="y"):
    if isinstance(y, pd.Series):
        return y
    return pd.Series(np.asarray(y).ravel(), name=name)

# If you already have 'feature_names' from your load step, pass it here.
# Otherwise this will auto-create X0..Xn-1 or pull names from the dataset loader.
X_train = ensure_df(X_train)            # or ensure_df(X_train, feature_names)
X_test  = ensure_df(X_test)             # or ensure_df(X_test,  feature_names)
y_train = ensure_series(y_train, "y")
y_test  = ensure_series(y_test,  "y")

print(X_train.shape, X_test.shape, y_train.shape)
print("Columns:", list(X_train.columns))

(16512, 8) (4128, 8) (16512,)
Columns: ['MedInc', 'HouseAge', 'AveRooms', 'AveBedrms', 'Population', 'AveOccup', 'Latitude', 'Longitude']


In [7]:
# Cell A: helpers — run once
import time, pickle, os
from pathlib import Path
import numpy as np, pandas as pd

def ensure_df(X, feature_names=None):
    """Return a pandas DataFrame (if already DF returns it)."""
    if isinstance(X, pd.DataFrame):
        return X
    if feature_names is None:
        try:
            from sklearn.datasets import fetch_california_housing
            feature_names = fetch_california_housing(as_frame=True).feature_names
        except Exception:
            feature_names = [f"X{i}" for i in range(X.shape[1])]
    return pd.DataFrame(np.asarray(X), columns=feature_names)

def ensure_series(y, name="y"):
    if isinstance(y, pd.Series):
        return y
    return pd.Series(np.asarray(y).ravel(), name=name)

def safe_makedir(p: Path):
    p.mkdir(parents=True, exist_ok=True)

def save_pickle(obj, path):
    safe_makedir(Path(path).parent)
    with open(path, "wb") as f:
        pickle.dump(obj, f)

def save_parquet(df, path):
    safe_makedir(Path(path).parent)
    # if df is numpy, convert
    if not isinstance(df, pd.DataFrame):
        df = pd.DataFrame(df)
    df.to_parquet(path)

def save_series_parquet(s, path):
    safe_makedir(Path(path).parent)
    if not isinstance(s, pd.Series):
        s = pd.Series(s)
    s.to_frame().to_parquet(path)

print("helpers ready")

helpers ready


In [8]:
# CELL: patch add_sr_features (backwards-compatible) + helper to find latest artifact folder for K
import pickle, json, re
from pathlib import Path
import pandas as pd
import numpy as np

def add_sr_features_robust(X_df, sr_bundle):
    """Robust version: accepts sr_bundle values as either:
       - (est, cols)
       - dict with keys 'est' and 'cols' (possibly additional metadata)
    Returns X_df augmented with SR columns (keeps original X_df).
    """
    X_aug = X_df.copy()
    for name, val in sr_bundle.items():
        # normalize
        if isinstance(val, tuple) or isinstance(val, list):
            est, cols = val[0], list(val[1])
        elif isinstance(val, dict):
            est, cols = val.get("est"), list(val.get("cols", []))
        else:
            raise ValueError(f"Unsupported model bundle type for {name}: {type(val)}")
        if est is None or cols is None:
            print(f"Skipping {name}: no est/cols")
            continue
        # ensure DataFrame slice
        X_slice = X_aug[cols]
        # predict (wrap in try)
        try:
            X_aug[name] = est.predict(X_slice)
        except Exception as e:
            print(f"Prediction failed for {name}: {e}")
            # create a column of NaNs so shapes don't break
            X_aug[name] = np.nan
    return X_aug

def find_latest_artifact_for_K(artifacts_root="artifacts", K=3):
    root = Path(artifacts_root)
    if not root.exists():
        raise FileNotFoundError(f"artifacts root not found: {root}")
    # match dirs that have _K{K}_ in their name, sort by modified time
    dirs = [d for d in root.iterdir() if d.is_dir() and re.search(rf"_K{K}[_\b]", d.name)]
    if not dirs:
        # fallback: match any dir containing K{K}
        dirs = [d for d in root.iterdir() if d.is_dir() and f"K{K}" in d.name]
    if not dirs:
        raise FileNotFoundError(f"No artifact dirs found for K={K} in {root}")
    dirs_sorted = sorted(dirs, key=lambda d: d.stat().st_mtime, reverse=True)
    return dirs_sorted[0]

print("Patch loaded — add_sr_features_robust ready. Use find_latest_artifact_for_K('artifacts', K).")

Patch loaded — add_sr_features_robust ready. Use find_latest_artifact_for_K('artifacts', K).


In [7]:
# --- Robust replacement: add_sr_features + run_strategy2_for_one_K ---
import inspect, pickle, pathlib, time, traceback, json
import numpy as np, pandas as pd

# --- Robust add_sr_features (handles multiple sr_bundle formats) ---
def add_sr_features(X_df, sr_bundle):
    """
    X_df: pandas DataFrame
    sr_bundle: mapping-like (key->value) where value may be:
      - (estimator, cols)
      - {'est': estimator, 'cols': cols}
      - estimator (if cols encoded elsewhere) -> in this case we will skip (can't predict)
    Returns a new DataFrame with SR columns appended. Column names are inferred from keys.
    """
    X_aug = X_df.copy()
    # Accept numpy arrays as well
    if not isinstance(X_aug, pd.DataFrame):
        try:
            X_aug = pd.DataFrame(X_aug, columns=[f"X{i}" for i in range(X_aug.shape[1])])
        except Exception:
            X_aug = pd.DataFrame(np.asarray(X_aug))
    for name, v in sr_bundle.items() if hasattr(sr_bundle, "items") else enumerate(sr_bundle):
        # Normalize (name, value)
        if not isinstance(name, str):
            # sr_bundle was a list/tuple of (name, value) items
            try:
                name, v = v
            except Exception:
                # can't interpret — skip
                continue
        # resolve estimator and cols
        est = None
        cols = None
        if isinstance(v, dict):
            est = v.get("est") or v.get("estimator") or v.get("estimator_obj") or v.get("estimator_obj", None)
            cols = v.get("cols") or v.get("columns") or v.get("features") or v.get("cols", None)
        elif isinstance(v, (list, tuple)):
            # (est, cols) or (est, cols, extras)
            if len(v) >= 2:
                est = v[0]
                cols = v[1]
            elif len(v) == 1:
                est = v[0]
        else:
            # Could be an estimator alone
            est = v
        if est is None:
            # Nothing sensible to call
            continue
        # If cols is None, try to use full X_df columns
        if cols is None:
            cols = list(X_aug.columns)
        # If cols are indices, convert to names
        try:
            # if integer indices given, map to column names
            if all(isinstance(c, (int, np.integer)) for c in cols):
                cols = [X_aug.columns[int(c)] for c in cols]
        except Exception:
            pass
        # Now attempt prediction
        try:
            # Many SR estimators use .predict on array-like input
            X_sub = X_aug[cols]
            pred = est.predict(X_sub)
            # ensure 1d array
            pred = np.asarray(pred).ravel()
            X_aug[name] = pred
        except Exception as e:
            # Save a warning column and skip
            warn_col = f"{name}__FAILED"
            X_aug[warn_col] = np.nan
            # keep trace in a log file later (caller can inspect)
            print(f"Warning: couldn't predict SR {name}: {e}")
            continue
    return X_aug

# --- Robust run_strategy2_for_one_K replacement ---
def run_strategy2_for_one_K(X_train, X_test, y_train, y_test,
                           K,
                           fs_name="base",
                           funcset=None,
                           gen_grid=(50,100,200),
                           parsimony_grid=(0.001,0.005,0.01),
                           population_size=1000,
                           out_root="artifacts",
                           random_state=42):
    """
    A robust wrapper that runs the Auto-Sweep for a single K.
    Expects the notebook to define:
      - make_feature_groups(X, K) or uses the local implementation
      - fit_sr_one_group_grid(Xg, y, gen_grid=..., parsimony_grid=..., function_set=..., random_state=...) -> returns dict with 'est' or similar
      - eval_models(Xtr, Xte, ytr, yte)
    This wrapper will:
      - iterate groups
      - call fit helper with/without population_size depending on signature
      - persist partial state after each group
      - save final artifacts (sr_models, X*_aug, summary.csv)
    """
    # ensure work dir
    out_root = pathlib.Path(out_root)
    out_root.mkdir(parents=True, exist_ok=True)

    # create run id
    ts = time.strftime("%Y%m%d_%H%M%S")
    run_id = f"strategy2_california_K{K}_{fs_name}_{ts}"
    outdir = out_root / run_id
    outdir.mkdir(parents=True, exist_ok=True)

    # prepare safe references to helpers from the notebook scope
    # prefer notebook-defined functions, fallback to local ones if present
    local_ns = globals()

    # find make_feature_groups
    mf = None
    if "make_feature_groups" in local_ns:
        mf = local_ns["make_feature_groups"]
    else:
        # fallback to simple clustering function from your notebook if not present
        from sklearn.cluster import AgglomerativeClustering
        def mf(X, K_):
            C = np.corrcoef(X, rowvar=False)
            D = 1.0 - np.abs(C)
            np.fill_diagonal(D, 0.0)
            model = AgglomerativeClustering(metric="precomputed", linkage="average", n_clusters=K_)
            labels = model.fit_predict(D)
            return [sorted(np.where(labels==g)[0].tolist()) for g in range(K_)]
    # find fit_sr_one_group_grid
    fit_helper = local_ns.get("fit_sr_one_group_grid")
    if fit_helper is None:
        raise RuntimeError("fit_sr_one_group_grid not found in notebook. Please define it before calling run_strategy2_for_one_K.")

    # detect whether fit_helper accepts population_size
    fit_sig = inspect.signature(fit_helper)
    accepts_population_size = "population_size" in fit_sig.parameters

    # storage for results
    runs = []
    sr_models = {}   # collected SR models per SR column
    kept_cols = []

    t0_all = time.time()
    try:
        # ensure pandas DataFrame shapes
        if isinstance(X_train, np.ndarray):
            X_train_df = pd.DataFrame(X_train, columns=[f"X{i}" for i in range(X_train.shape[1])])
        else:
            X_train_df = X_train.copy()
        if isinstance(X_test, np.ndarray):
            X_test_df = pd.DataFrame(X_test, columns=X_train_df.columns)
        else:
            X_test_df = X_test.copy()

        # group features
        groups = mf(X_train_df.values if hasattr(X_train_df, "values") else X_train_df, K)
        print(f"[K={K} | {fs_name}] Groups: {[[X_train_df.columns[i] for i in g] for g in groups]}")

        # function_set object: if funcset is None, try to lookup function_sets[fs_name]
        if funcset is None:
            funcset = local_ns.get("function_sets", {}).get(fs_name, None)
        if funcset is None:
            print("No function_set object found; proceeding with funcset as passed.")

        # loop groups
        for gi, idxs in enumerate(groups, start=1):
            print(f"[K={K}] Processing group {gi}/{len(groups)} (cols {idxs}) ...")
            Xg = X_train_df.iloc[:, idxs]
            yg = np.asarray(y_train).ravel()

            # Call fit helper with or without population_size depending on signature
            kwargs = dict(gen_grid=gen_grid, parsimony_grid=parsimony_grid, function_set=funcset, random_state=random_state)
            try:
                if accepts_population_size:
                    best = fit_helper(Xg, yg, population_size=population_size, **kwargs)
                else:
                    # call without population_size if not accepted
                    best = fit_helper(Xg, yg, **kwargs)
            except TypeError as e:
                # last-resort: try positional ordering (some versions expect Xg, y, gen_grid, parsimony_grid, function_set, random_state)
                try:
                    best = fit_helper(Xg, yg, gen_grid, parsimony_grid, funcset, random_state)
                except Exception as e2:
                    raise

            # best is expected to be a dict-like with 'est' or 'estimator' and 'cols' or similar
            # Normalize to mapping name->(est, cols)
            try:
                # attempt to get name
                sr_name = f"SR_G{gi}"
                if isinstance(best, dict) and ("est" in best or "estimator" in best):
                    est_obj = best.get("est") or best.get("estimator")
                    cols = best.get("cols") or idxs
                    sr_models[sr_name] = {"est": est_obj, "cols": cols}
                elif isinstance(best, tuple) or isinstance(best, list):
                    # assume (est, cols)
                    est_obj = best[0]
                    cols = best[1] if len(best) > 1 else idxs
                    sr_models[sr_name] = {"est": est_obj, "cols": cols}
                elif hasattr(best, "predict"):
                    # best is estimator object alone
                    sr_models[sr_name] = {"est": best, "cols": idxs}
                else:
                    # unknown format — skip
                    print(f"Unrecognized fit helper return for group {gi}; saved raw object under sr_models['{sr_name}'].")
                    sr_models[sr_name] = {"est": best, "cols": idxs}
            except Exception as e:
                print("Error normalizing 'best' from fit helper:", e)
                sr_models[f"SR_G{gi}"] = {"est": best, "cols": idxs}

            kept_cols.append(sr_name)

            # save partial so far
            partial = {
                "K": K,
                "fs_name": fs_name,
                "groups": groups,
                "current_group": gi,
                "sr_models": sr_models,
                "kept_cols": kept_cols,
                "timestamp": time.time()
            }
            partial_path = outdir / "partial_state.pkl"
            with open(partial_path, "wb") as f:
                pickle.dump(partial, f)
            print(f"Saved partial after group {gi} to {partial_path}")

        # After all groups fitted, add SR features and evaluate
        print("Adding SR features and evaluating final models...")
        Xtr_aug = add_sr_features(X_train_df.copy(), {k:v for k,v in sr_models.items()})
        Xte_aug = add_sr_features(X_test_df.copy(), {k:v for k,v in sr_models.items()})

        # eval_models should be defined in notebook
        eval_fn = local_ns.get("eval_models")
        if eval_fn is None:
            raise RuntimeError("eval_models not found in notebook namespace.")

        base = eval_fn(X_train_df, X_test_df, y_train, y_test)
        aug = eval_fn(Xtr_aug, Xte_aug, y_train, y_test)

        # assemble summary DataFrame as in your original code
        summary = pd.DataFrame({
            "Metric": ["LR_RMSE","Ridge_RMSE","XGB_RMSE"],
            "Baseline": [base["LR_RMSE"], base["Ridge_RMSE"], base["XGB_RMSE"]],
            "+SR": [aug["LR_RMSE"], aug["Ridge_RMSE"], aug["XGB_RMSE"]],
        })
        summary["Δ (Aug-Base)"] = summary["+SR"] - summary["Baseline"]
        score = float(summary["Δ (Aug-Base)"].mean())

        # persist results
        with open(outdir / "sr_models.pkl", "wb") as f:
            pickle.dump(sr_models, f)

        # Save augmented features as parquet if pandas present
        try:
            Xtr_aug.to_parquet(outdir / "X_train_aug.parquet")
            Xte_aug.to_parquet(outdir / "X_test_aug.parquet")
        except Exception as e:
            # fallback to pickle
            with open(outdir / "X_train_aug.pkl", "wb") as f:
                pickle.dump(Xtr_aug, f)
            with open(outdir / "X_test_aug.pkl", "wb") as f:
                pickle.dump(Xte_aug, f)

        # save summary CSV + JSON meta
        summary.to_csv(outdir / "summary.csv", index=False)
        meta = {
            "K": K,
            "fs_name": fs_name,
            "score_mean_delta": score,
            "kept_SR": kept_cols,
            "timestamp": time.time(),
        }
        with open(outdir / "meta.json", "w") as f:
            json.dump(meta, f, indent=2)

        print("\nSaved bundle to:", outdir, "| total time: %.1fs" % (time.time()-t0_all))

    except Exception as e:
        # save fatal exception partial
        try:
            trace = traceback.format_exc()
            (outdir / "fatal_exception.txt").write_text(trace)
            print("Fatal exception; partial state saved at:", outdir / "partial_state.pkl")
        except Exception:
            pass
        raise

    return str(outdir)

# ---- End of helper definitions ----
print("Patched run_strategy2_for_one_K and add_sr_features installed. You can now call your K-run cell.")

Patched run_strategy2_for_one_K and add_sr_features installed. You can now call your K-run cell.


In [30]:
# Run K=3 with the "rich" function set
K = 4
fs_name = "base"

# try to grab the rich function-set object from your notebook (recommended)
try:
    funcset = function_sets[fs_name]
    print("Using function_sets['rich'] from notebook.")
except Exception as e:
    # fallback: use literal names (best-effort). If your fit helper expects make_function objects (LOG, SQRT),
    # you must ensure function_sets['rich'] exists; otherwise the fallback uses plain names.
    print("function_sets['rich'] not found; using fallback tuple of names.")
    funcset = ('add','sub','mul','div','sqrt','log','sin','cos')

# adjust these params if you want (population_size smaller -> faster but less thorough)
outdir = run_strategy2_for_one_K(
    X_train=X_train, X_test=X_test, y_train=y_train, y_test=y_test,
    K=K,
    fs_name=fs_name,
    funcset=funcset,
    gen_grid=(50,100,200),
    parsimony_grid=(0.001,0.005,0.01),
    population_size=1000,
    out_root="artifacts",
    random_state=42
)

print("K=3 run started / completed. Artifacts stored at:", outdir)

Using function_sets['rich'] from notebook.
[K=4 | base] Groups: [['AveOccup'], ['HouseAge', 'Population'], ['Latitude', 'Longitude'], ['MedInc', 'AveRooms', 'AveBedrms']]
[K=4] Processing group 1/4 (cols [5]) ...
Saved partial after group 1 to artifacts/strategy2_california_K4_base_20251224_191521/partial_state.pkl
[K=4] Processing group 2/4 (cols [1, 4]) ...
Saved partial after group 2 to artifacts/strategy2_california_K4_base_20251224_191521/partial_state.pkl
[K=4] Processing group 3/4 (cols [6, 7]) ...
Saved partial after group 3 to artifacts/strategy2_california_K4_base_20251224_191521/partial_state.pkl
[K=4] Processing group 4/4 (cols [0, 2, 3]) ...
Saved partial after group 4 to artifacts/strategy2_california_K4_base_20251224_191521/partial_state.pkl
Adding SR features and evaluating final models...

Saved bundle to: artifacts/strategy2_california_K4_base_20251224_191521 | total time: 2980.1s
K=3 run started / completed. Artifacts stored at: artifacts/strategy2_california_K4_base

In [ ]:
# Run K=3 with the "rich" function set
K = 4
fs_name = "rich"

# try to grab the rich function-set object from your notebook (recommended)
try:
    funcset = function_sets[fs_name]
    print("Using function_sets['rich'] from notebook.")
except Exception as e:
    # fallback: use literal names (best-effort). If your fit helper expects make_function objects (LOG, SQRT),
    # you must ensure function_sets['rich'] exists; otherwise the fallback uses plain names.
    print("function_sets['rich'] not found; using fallback tuple of names.")
    funcset = ('add','sub','mul','div','sqrt','log','sin','cos')

# adjust these params if you want (population_size smaller -> faster but less thorough)
outdir = run_strategy2_for_one_K(
    X_train=X_train, X_test=X_test, y_train=y_train, y_test=y_test,
    K=K,
    fs_name=fs_name,
    funcset=funcset,
    gen_grid=(50,100,200),
    parsimony_grid=(0.001,0.005,0.01),
    population_size=1000,
    out_root="artifacts",
    random_state=42
)

print("K=3 run started / completed. Artifacts stored at:", outdir)

Using function_sets['rich'] from notebook.
[K=4 | rich] Groups: [['AveOccup'], ['HouseAge', 'Population'], ['Latitude', 'Longitude'], ['MedInc', 'AveRooms', 'AveBedrms']]
[K=4] Processing group 1/4 (cols [5]) ...


Testing

In [9]:
# Cell A: list artifacts and recent partials
import pathlib, json, pickle, os
root = pathlib.Path("artifacts")
if not root.exists():
    print("No artifacts/ folder found in CWD:", pathlib.Path.cwd())
else:
    runs = sorted([p for p in root.iterdir() if p.is_dir()], key=lambda p: p.stat().st_mtime, reverse=True)
    print(f"Found {len(runs)} run directories under {root}:")
    for r in runs[:20]:
        print(" -", r.name)
        # print small summary of files inside
        files = list(r.glob("*"))
        names = [f.name for f in files]
        print("    files:", names)
# show any immediate partial files at top-level
partials = list(root.glob("**/partial_state.pkl"))
print("\nFound partial_state.pkl files:", len(partials))
for p in partials[:10]:
    print(" -", p)

Found 15 run directories under artifacts:
 - strategy2_california_K5_base_20251227_112609
    files: []
 - .ipynb_checkpoints
    files: []
 - strategy2_california_K5_rich_20251226_200600
    files: ['X_test_aug.parquet', 'summary.csv', 'partial_state.pkl', 'meta.json', 'X_train_aug.parquet', 'sr_models.pkl']
 - strategy2_california_K5_base_20251226_174852
    files: ['X_test_aug.parquet', 'summary.csv', 'partial_state.pkl', 'meta.json', 'X_train_aug.parquet', 'sr_models.pkl']
 - strategy2_california_K5_base_20251226_171347
    files: []
 - strategy2_california_K5_base_20251226_171228
    files: []
 - strategy2_california_K5_base_20251226_171113
    files: []
 - strategy2_california_K5_base_20251226_171058
    files: []
 - strategy2_california_K4_rich_20251224_201259
    files: ['X_test_aug.parquet', 'summary.csv', 'partial_state.pkl', 'meta.json', 'X_train_aug.parquet', 'sr_models.pkl']
 - strategy2_california_K4_base_20251224_191521
    files: ['X_test_aug.parquet', 'summary.csv', 'p

In [10]:
# A0 — inspect artifacts
from pathlib import Path
root = Path("artifacts")
print("Artifacts root:", root.resolve())
if not root.exists():
    print("No artifacts directory yet.")
else:
    runs = sorted([d for d in root.iterdir() if d.is_dir()], key=lambda d: d.stat().st_mtime, reverse=True)
    for d in runs:
        files = list(d.iterdir())
        print(d.name, " | files:", [p.name for p in files])
    # show which runs have partial_state.pkl
    partials = [d for d in runs if (d/"partial_state.pkl").exists()]
    print("\nFound partial_state.pkl files:", len(partials))
    for p in partials[:10]:
        print(" -", p.name, "| partial:", (p/"partial_state.pkl").exists(), "| fatal:", (p/"fatal_exception.txt").exists())
        if (p/"fatal_exception.txt").exists():
            print("   -> fatal_exception.txt preview:")
            print((p/"fatal_exception.txt").read_text()[:800])

Artifacts root: /home/rshayan1/artifacts
strategy2_california_K5_base_20251227_112609  | files: []
.ipynb_checkpoints  | files: []
strategy2_california_K5_rich_20251226_200600  | files: ['X_test_aug.parquet', 'summary.csv', 'partial_state.pkl', 'meta.json', 'X_train_aug.parquet', 'sr_models.pkl']
strategy2_california_K5_base_20251226_174852  | files: ['X_test_aug.parquet', 'summary.csv', 'partial_state.pkl', 'meta.json', 'X_train_aug.parquet', 'sr_models.pkl']
strategy2_california_K5_base_20251226_171347  | files: []
strategy2_california_K5_base_20251226_171228  | files: []
strategy2_california_K5_base_20251226_171113  | files: []
strategy2_california_K5_base_20251226_171058  | files: []
strategy2_california_K4_rich_20251224_201259  | files: ['X_test_aug.parquet', 'summary.csv', 'partial_state.pkl', 'meta.json', 'X_train_aug.parquet', 'sr_models.pkl']
strategy2_california_K4_base_20251224_191521  | files: ['X_test_aug.parquet', 'summary.csv', 'partial_state.pkl', 'meta.json', 'X_train_

In [11]:
# A1 — inspect a partial state (edit path if you want a different run)
import pickle, json
p = "artifacts/strategy2_california_K3_base_20251224_160434/partial_state.pkl"
from pathlib import Path
if Path(p).exists():
    st = pickle.load(open(p,"rb"))
    print("keys:", list(st.keys()))
    import pprint; pprint.pprint(st, width=120)
else:
    print("Partial not found at:", p)

keys: ['sr_models', 'kept_cols', 'runs']
{'kept_cols': ['SR_G1', 'SR_G2', 'SR_G3'],
 'runs': [],
 'sr_models': {'SR_G1': {'cols': ['MedInc', 'HouseAge', 'AveRooms', 'AveBedrms', 'Population'],
                         'est': SymbolicRegressor(generations=50, parsimony_coefficient=0.005, random_state=42),
                         'gens': 50,
                         'pc': 0.005,
                         'val_rmse': 0.8537540994341871},
               'SR_G2': {'cols': ['AveOccup'],
                         'est': SymbolicRegressor(generations=200, parsimony_coefficient=0.005, random_state=42),
                         'gens': 200,
                         'pc': 0.005,
                         'val_rmse': 1.1206298117582814},
               'SR_G3': {'cols': ['Latitude', 'Longitude'],
                         'est': SymbolicRegressor(generations=50, parsimony_coefficient=0.01, random_state=42),
                         'gens': 50,
                         'pc': 0.01,
                    

In [12]:
# RESUME helper: continue a partially completed run (pick latest if not provided)
import inspect, pickle, json, time, traceback
from pathlib import Path
import numpy as np, pandas as pd

def resume_partial_run(K=None, fs_name=None, artifacts_root="artifacts",
                       population_size=1000, gen_grid=(50,100,200), parsimony_grid=(0.001,0.005,0.01),
                       random_state=42, run_dir_override=None):
    """
    Resume a partial run previously created by run_strategy2_for_one_K.
    - If run_dir_override is given, resume that exact folder.
    - Otherwise finds the most recent artifacts folder matching K and fs_name that contains partial_state.pkl.
    """
    root = Path(artifacts_root)
    assert root.exists(), f"Artifacts root not found: {root}"
    # choose run folder
    if run_dir_override:
        outdir = Path(run_dir_override)
        if not outdir.exists():
            raise FileNotFoundError("Provided run_dir_override not found: " + str(outdir))
    else:
        # gather candidates
        cand = []
        for d in root.iterdir():
            if not d.is_dir(): 
                continue
            if (d/"partial_state.pkl").exists():
                name = d.name
                if K is not None and f"_K{K}_" not in name and f"_K{K}" not in name:
                    continue
                if fs_name is not None and f"_{fs_name}_" not in name and not name.endswith(f"_{fs_name}"):
                    continue
                cand.append(d)
        if not cand:
            raise FileNotFoundError("No candidate partial runs found for given K/fs_name in " + str(root))
        # newest first
        cand = sorted(cand, key=lambda d: d.stat().st_mtime, reverse=True)
        outdir = cand[0]

    print("Resuming:", outdir)

    # load partial
    partial_path = outdir / "partial_state.pkl"
    st = pickle.load(open(partial_path, "rb"))
    groups = st.get("groups")
    sr_models = st.get("sr_models", {}) or {}
    kept_cols = st.get("kept_cols", []) or []
    current_group = st.get("current_group", 0)
    # infer last completed index if current_group not present: use len(kept_cols)
    last_done = int(current_group) if current_group else len(kept_cols)
    print(f"Loaded partial: last_done={last_done}, total groups={len(groups)}")
    # determine remaining groups (indices)
    remaining = []
    for i, idxs in enumerate(groups, start=1):
        if i > last_done:
            remaining.append((i, idxs))
    if not remaining:
        print("Nothing to do: partial indicates all groups are completed. Finalizing artifacts if needed...")
    # ensure helpers exist
    local_ns = globals()
    fit_helper = local_ns.get("fit_sr_one_group_grid")
    if fit_helper is None:
        raise RuntimeError("fit_sr_one_group_grid not found in notebook. Cannot resume.")
    eval_fn = local_ns.get("eval_models")
    if eval_fn is None:
        raise RuntimeError("eval_models not found in notebook. Cannot finalize.")
    # detect signature for population_size
    fit_sig = inspect.signature(fit_helper)
    accepts_population_size = "population_size" in fit_sig.parameters

    # we need X_train, X_test, y_train, y_test in notebook namespace
    Xtr = local_ns.get("X_train")
    Xte = local_ns.get("X_test")
    ytr = local_ns.get("y_train")
    yte = local_ns.get("y_test")
    if Xtr is None or Xte is None or ytr is None or yte is None:
        raise RuntimeError("X_train / X_test / y_train / y_test not found in notebook namespace.")

    # Convert to DataFrame if necessary (use your ensure_df if present)
    ensure_df_fn = local_ns.get("ensure_df")
    if ensure_df_fn is None:
        def ensure_df_local(X):
            if isinstance(X, pd.DataFrame):
                return X.copy()
            return pd.DataFrame(np.asarray(X))
        ensure_df_fn = ensure_df_local

    Xtr_df = ensure_df_fn(Xtr)
    Xte_df = ensure_df_fn(Xte)

    try:
        # loop remaining
        for gi, idxs in remaining:
            print(f"=== Resuming: fitting group {gi}/{len(groups)} (cols {idxs}) ===")
            Xg = Xtr_df.iloc[:, idxs]
            yg = np.asarray(ytr).ravel()
            kwargs = dict(gen_grid=gen_grid, parsimony_grid=parsimony_grid,
                          function_set=st.get("fs_name", None) or None, random_state=random_state)
            # If original run saved a function-set object, try to use notebook function_sets
            # attempt to find function_set object if available
            try:
                fsname = st.get("fs_name")
                if fsname and "function_sets" in local_ns and fsname in local_ns["function_sets"]:
                    kwargs["function_set"] = local_ns["function_sets"][fsname]
            except Exception:
                pass

            try:
                if accepts_population_size:
                    best = fit_helper(Xg, yg, population_size=population_size, **kwargs)
                else:
                    best = fit_helper(Xg, yg, **kwargs)
            except TypeError as e:
                # try positional fallback
                try:
                    best = fit_helper(Xg, yg, gen_grid, parsimony_grid, kwargs.get("function_set", None), random_state)
                except Exception as e2:
                    raise

            # normalize 'best' into sr_models entry (same logic as your runner)
            sr_name = f"SR_G{gi}"
            try:
                if isinstance(best, dict) and ("est" in best or "estimator" in best):
                    est_obj = best.get("est") or best.get("estimator")
                    cols = best.get("cols") or idxs
                    sr_models[sr_name] = {"est": est_obj, "cols": cols}
                elif isinstance(best, (list,tuple)):
                    est_obj = best[0]
                    cols = best[1] if len(best) > 1 else idxs
                    sr_models[sr_name] = {"est": est_obj, "cols": cols}
                elif hasattr(best, "predict"):
                    sr_models[sr_name] = {"est": best, "cols": idxs}
                else:
                    sr_models[sr_name] = {"est": best, "cols": idxs}
            except Exception as e:
                print("Normalization error; saving raw best:", e)
                sr_models[sr_name] = {"est": best, "cols": idxs}

            kept_cols.append(sr_name)
            last_done = gi
            # update + save partial
            partial = dict(K=K, fs_name=fs_name or st.get("fs_name"), groups=groups,
                           current_group=last_done, sr_models=sr_models, kept_cols=kept_cols, timestamp=time.time())
            with open(partial_path, "wb") as f:
                pickle.dump(partial, f)
            print(f"Saved partial after group {gi} -> {partial_path}")

        # after all groups done -> finalize: add SR features and eval
        print("All groups fitted. Adding SR features and evaluating final models...")
        # pick robust add_sr function from namespace
        add_sr_fn = local_ns.get("add_sr_features") or local_ns.get("add_sr_features_robust")
        if add_sr_fn is None:
            # fallback to local simple predictor
            def add_sr_fallback(X_df, sr_bundle):
                X_aug = X_df.copy()
                for name, val in sr_bundle.items():
                    try:
                        if isinstance(val, (list,tuple)):
                            est, cols = val[0], val[1]
                        elif isinstance(val, dict):
                            est, cols = val.get("est"), val.get("cols")
                        else:
                            est, cols = val, X_df.columns.tolist()
                        X_aug[name] = est.predict(X_df[cols])
                    except Exception:
                        X_aug[name] = np.nan
                return X_aug
            add_sr_fn = add_sr_fallback

        Xtr_aug = add_sr_fn(Xtr_df.copy(), sr_models)
        Xte_aug = add_sr_fn(Xte_df.copy(), sr_models)

        base = eval_fn(Xtr_df, Xte_df, ytr, yte)
        aug = eval_fn(Xtr_aug, Xte_aug, ytr, yte)

        summary = pd.DataFrame({
            "Metric": ["LR_RMSE","Ridge_RMSE","XGB_RMSE"],
            "Baseline": [base["LR_RMSE"], base["Ridge_RMSE"], base["XGB_RMSE"]],
            "+SR": [aug["LR_RMSE"], aug["Ridge_RMSE"], aug["XGB_RMSE"]],
        })
        summary["Δ (Aug-Base)"] = summary["+SR"] - summary["Baseline"]
        score = float(summary["Δ (Aug-Base)"].mean())

        # persist final artifacts
        with open(outdir / "sr_models.pkl", "wb") as f:
            pickle.dump(sr_models, f)
        try:
            Xtr_aug.to_parquet(outdir / "X_train_aug.parquet")
            Xte_aug.to_parquet(outdir / "X_test_aug.parquet")
        except Exception:
            with open(outdir / "X_train_aug.pkl", "wb") as f:
                pickle.dump(Xtr_aug, f)
            with open(outdir / "X_test_aug.pkl", "wb") as f:
                pickle.dump(Xte_aug, f)
        summary.to_csv(outdir / "summary.csv", index=False)
        meta = {"K": K, "fs_name": fs_name or st.get("fs_name"), "score_mean_delta": score,
                "kept_SR": kept_cols, "timestamp": time.time()}
        with open(outdir / "meta.json", "w") as f:
            json.dump(meta, f, indent=2)
        print("Finalized and saved artifacts to:", outdir)
        return str(outdir)

    except Exception as e:
        trace = traceback.format_exc()
        try:
            (outdir / "fatal_exception.txt").write_text(trace)
            print("Fatal exception saved to:", outdir / "fatal_exception.txt")
        except Exception:
            pass
        raise

# Example usage:
# resume_partial_run(K=5, fs_name="base")
# or resume_partial_run(run_dir_override="artifacts/strategy2_california_K5_base_20251226_171347")
print("Resume helper loaded — call resume_partial_run(...) with desired K or run_dir_override.")

Resume helper loaded — call resume_partial_run(...) with desired K or run_dir_override.


In [13]:
# A0 — inspect artifacts
from pathlib import Path
root = Path("artifacts")
print("Artifacts root:", root.resolve())
if not root.exists():
    print("No artifacts directory yet.")
else:
    runs = sorted([d for d in root.iterdir() if d.is_dir()], key=lambda d: d.stat().st_mtime, reverse=True)
    for d in runs:
        files = list(d.iterdir())
        print(d.name, " | files:", [p.name for p in files])
    # show which runs have partial_state.pkl
    partials = [d for d in runs if (d/"partial_state.pkl").exists()]
    print("\nFound partial_state.pkl files:", len(partials))
    for p in partials[:10]:
        print(" -", p.name, "| partial:", (p/"partial_state.pkl").exists(), "| fatal:", (p/"fatal_exception.txt").exists())
        if (p/"fatal_exception.txt").exists():
            print("   -> fatal_exception.txt preview:")
            print((p/"fatal_exception.txt").read_text()[:800])

Artifacts root: /home/rshayan1/artifacts
strategy2_california_K5_base_20251227_112609  | files: []
.ipynb_checkpoints  | files: []
strategy2_california_K5_rich_20251226_200600  | files: ['X_test_aug.parquet', 'summary.csv', 'partial_state.pkl', 'meta.json', 'X_train_aug.parquet', 'sr_models.pkl']
strategy2_california_K5_base_20251226_174852  | files: ['X_test_aug.parquet', 'summary.csv', 'partial_state.pkl', 'meta.json', 'X_train_aug.parquet', 'sr_models.pkl']
strategy2_california_K5_base_20251226_171347  | files: []
strategy2_california_K5_base_20251226_171228  | files: []
strategy2_california_K5_base_20251226_171113  | files: []
strategy2_california_K5_base_20251226_171058  | files: []
strategy2_california_K4_rich_20251224_201259  | files: ['X_test_aug.parquet', 'summary.csv', 'partial_state.pkl', 'meta.json', 'X_train_aug.parquet', 'sr_models.pkl']
strategy2_california_K4_base_20251224_191521  | files: ['X_test_aug.parquet', 'summary.csv', 'partial_state.pkl', 'meta.json', 'X_train_

In [14]:
# 1. Quick sanity checks
import inspect, pathlib, pickle, json, time
from pathlib import Path

print("cwd:", Path.cwd())
print("Has run_strategy2_for_one_K:", "run_strategy2_for_one_K" in globals())
print("Has resume_partial_run:", "resume_partial_run" in globals())
print("Has fit_sr_one_group_grid:", "fit_sr_one_group_grid" in globals())
print("Has eval_models:", "eval_models" in globals())
print("Artifacts root exists:", Path("artifacts").exists())
# list recent artifacts
root = Path("artifacts")
if root.exists():
    for d in sorted([x for x in root.iterdir() if x.is_dir()], key=lambda x: x.stat().st_mtime, reverse=True)[:12]:
        print("-", d.name, " | files:", [p.name for p in d.iterdir()])
else:
    print("No artifacts directory found yet.")

cwd: /home/rshayan1
Has run_strategy2_for_one_K: True
Has resume_partial_run: True
Has fit_sr_one_group_grid: False
Has eval_models: False
Artifacts root exists: True
- strategy2_california_K5_base_20251227_112609  | files: []
- .ipynb_checkpoints  | files: []
- strategy2_california_K5_rich_20251226_200600  | files: ['X_test_aug.parquet', 'summary.csv', 'partial_state.pkl', 'meta.json', 'X_train_aug.parquet', 'sr_models.pkl']
- strategy2_california_K5_base_20251226_174852  | files: ['X_test_aug.parquet', 'summary.csv', 'partial_state.pkl', 'meta.json', 'X_train_aug.parquet', 'sr_models.pkl']
- strategy2_california_K5_base_20251226_171347  | files: []
- strategy2_california_K5_base_20251226_171228  | files: []
- strategy2_california_K5_base_20251226_171113  | files: []
- strategy2_california_K5_base_20251226_171058  | files: []
- strategy2_california_K4_rich_20251224_201259  | files: ['X_test_aug.parquet', 'summary.csv', 'partial_state.pkl', 'meta.json', 'X_train_aug.parquet', 'sr_model

In [15]:
# CELL: define eval_models
import numpy as np
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.metrics import mean_squared_error
import warnings

def eval_models(X_train, X_test, y_train, y_test, xgb_params=None):
    """
    Trains baseline models on the provided features and returns RMSEs:
      returns dict with keys: LR_RMSE, Ridge_RMSE, XGB_RMSE
    Uses XGBoost if available, otherwise sklearn.ensemble.GradientBoostingRegressor.
    """
    Xtr = X_train.copy() if hasattr(X_train, "copy") else X_train
    Xte = X_test.copy() if hasattr(X_test, "copy") else X_test
    ytr = np.asarray(y_train).ravel()
    yte = np.asarray(y_test).ravel()

    out = {}
    # Linear Regression
    try:
        lr = LinearRegression()
        lr.fit(Xtr, ytr)
        pred = lr.predict(Xte)
        out["LR_RMSE"] = float(np.sqrt(mean_squared_error(yte, pred)))
    except Exception as e:
        warnings.warn(f"LR failed: {e}")
        out["LR_RMSE"] = float("nan")

    # Ridge
    try:
        rd = Ridge()
        rd.fit(Xtr, ytr)
        pred = rd.predict(Xte)
        out["Ridge_RMSE"] = float(np.sqrt(mean_squared_error(yte, pred)))
    except Exception as e:
        warnings.warn(f"Ridge failed: {e}")
        out["Ridge_RMSE"] = float("nan")

    # XGB (fallback to GradientBoosting if not available)
    try:
        try:
            import xgboost as xgb
            XGBReg = xgb.XGBRegressor
            # default params — allow override via xgb_params
            params = dict(n_estimators=100, verbosity=0, n_jobs=1)
            if xgb_params:
                params.update(xgb_params)
            xgbm = XGBReg(**params)
        except Exception:
            # fallback
            from sklearn.ensemble import GradientBoostingRegressor
            XGBReg = GradientBoostingRegressor
            xgbm = XGBReg(n_estimators=100)
        xgbm.fit(Xtr, ytr)
        pred = xgbm.predict(Xte)
        out["XGB_RMSE"] = float(np.sqrt(mean_squared_error(yte, pred)))
    except Exception as e:
        warnings.warn(f"XGB/GBR failed: {e}")
        out["XGB_RMSE"] = float("nan")

    return out

print("eval_models defined (LR, Ridge, XGB/GBR fallback).")

eval_models defined (LR, Ridge, XGB/GBR fallback).


In [16]:
# CELL: define fit_sr_one_group_grid
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error
import warnings

def fit_sr_one_group_grid(Xg, y, gen_grid=(50,100,200), parsimony_grid=(0.001,0.005,0.01),
                          function_set=None, population_size=1000, random_state=42, max_time_per_model=None):
    """
    Grid-search wrapper for a single-group SR run.
    - Xg: DataFrame or array with the group's columns
    - y: target array-like
    - gen_grid, parsimony_grid: iterables
    - function_set: passed to SymbolicRegressor if not None
    - population_size: forwarded to SymbolicRegressor
    Returns: dict with keys: 'est' (fitted estimator), 'cols' (list), 'gens', 'pc', 'val_rmse'
    """
    # lazy import and checks
    try:
        from gplearn.genetic import SymbolicRegressor
    except Exception as e:
        raise RuntimeError("gplearn not available in this session. Please install gplearn==0.4.2 or adjust env.") from e

    Xg_df = Xg.copy() if hasattr(Xg, "copy") else Xg
    cols = list(Xg_df.columns) if hasattr(Xg_df, "columns") else [f"X{i}" for i in range(np.asarray(Xg_df).shape[1])]
    X_arr = np.asarray(Xg_df)
    y_arr = np.asarray(y).ravel()

    # small val split to choose best SR model
    trX, valX, trY, valY = train_test_split(X_arr, y_arr, test_size=0.2, random_state=random_state)

    best = {"val_rmse": float("inf"), "est": None, "gens": None, "pc": None, "cols": cols}
    tried = 0
    for gens in gen_grid:
        for pc in parsimony_grid:
            tried += 1
            try:
                sr_kwargs = dict(population_size=population_size,
                                 generations=int(gens),
                                 parsimony_coefficient=float(pc),
                                 random_state=int(random_state),
                                 verbose=0)
                # include function_set only if provided (and compatible)
                if function_set is not None:
                    # gplearn expects function_set as either string list or object created by gplearn.functions
                    sr_kwargs["function_set"] = function_set

                sr = SymbolicRegressor(**sr_kwargs)
                sr.fit(trX, trY)
                pred = sr.predict(valX)
                rmse = float(np.sqrt(mean_squared_error(valY, pred)))
                # store if improved
                if np.isfinite(rmse) and rmse < best["val_rmse"]:
                    best.update({"val_rmse": rmse, "est": sr, "gens": int(gens), "pc": float(pc)})
                print(f" tried gens={gens} pc={pc} → val_rmse={rmse:.4f}")
            except Exception as e:
                warnings.warn(f"SR fit failed for gens={gens} pc={pc}: {e}")
                continue

    if best["est"] is None:
        raise RuntimeError("No SymbolicRegressor succeeded on this group. See warnings above.")
    # return a dict compatible with your run wrapper
    out = {"est": best["est"], "cols": cols, "gens": best["gens"], "pc": best["pc"], "val_rmse": best["val_rmse"]}
    print(f"Best SR: gens={best['gens']} pc={best['pc']} val_rmse={best['val_rmse']:.4f}")
    return out

print("fit_sr_one_group_grid defined (gplearn SymbolicRegressor grid-search).")

fit_sr_one_group_grid defined (gplearn SymbolicRegressor grid-search).


Testing end

In [ ]:
# START K=5 base
outdir = run_strategy2_for_one_K(
    X_train=X_train, X_test=X_test, y_train=y_train, y_test=y_test,
    K=5,
    fs_name="base",
    funcset=None,
    gen_grid=(50,100,200),
    parsimony_grid=(0.001,0.005,0.01),
    population_size=1000,
    out_root="artifacts",
    random_state=42
)
print("Started run ->", outdir)

[K=5 | base] Groups: [['AveOccup'], ['MedInc'], ['HouseAge', 'Population'], ['AveRooms', 'AveBedrms'], ['Latitude', 'Longitude']]
No function_set object found; proceeding with funcset as passed.
[K=5] Processing group 1/5 (cols [5]) ...


In [ ]:
# START K=5 rich
outdir = run_strategy2_for_one_K(
    X_train=X_train, X_test=X_test, y_train=y_train, y_test=y_test,
    K=5,
    fs_name="rich",
    funcset=None,
    gen_grid=(50,100,200),
    parsimony_grid=(0.001,0.005,0.01),
    population_size=1000,
    out_root="artifacts",
    random_state=42
)
print("Started run ->", outdir)

[K=5 | rich] Groups: [['AveOccup'], ['MedInc'], ['HouseAge', 'Population'], ['AveRooms', 'AveBedrms'], ['Latitude', 'Longitude']]
No function_set object found; proceeding with funcset as passed.
[K=5] Processing group 1/5 (cols [5]) ...


In [19]:
# START K=6 base
outdir = run_strategy2_for_one_K(
    X_train=X_train, X_test=X_test, y_train=y_train, y_test=y_test,
    K=6,
    fs_name="base",
    funcset=None,
    gen_grid=(50,100,200),
    parsimony_grid=(0.001,0.005,0.01),
    population_size=1000,
    out_root="artifacts",
    random_state=42
)
print("Started run ->", outdir)

[K=6 | base] Groups: [['AveOccup'], ['Population'], ['MedInc'], ['HouseAge'], ['AveRooms', 'AveBedrms'], ['Latitude', 'Longitude']]
No function_set object found; proceeding with funcset as passed.
[K=6] Processing group 1/6 (cols [5]) ...
 tried gens=50 pc=0.001 → val_rmse=1.1551
 tried gens=50 pc=0.005 → val_rmse=1.2125
 tried gens=50 pc=0.01 → val_rmse=1.2125
 tried gens=100 pc=0.001 → val_rmse=1.1551
 tried gens=100 pc=0.005 → val_rmse=1.2125
 tried gens=100 pc=0.01 → val_rmse=1.2125
 tried gens=200 pc=0.001 → val_rmse=1.1468
 tried gens=200 pc=0.005 → val_rmse=1.1467
 tried gens=200 pc=0.01 → val_rmse=1.2125
Best SR: gens=200 pc=0.005 val_rmse=1.1467
Saved partial after group 1 to artifacts/strategy2_california_K6_base_20251227_144230/partial_state.pkl
[K=6] Processing group 2/6 (cols [4]) ...
 tried gens=50 pc=0.001 → val_rmse=1.2096
 tried gens=50 pc=0.005 → val_rmse=1.2118
 tried gens=50 pc=0.01 → val_rmse=1.2118
 tried gens=100 pc=0.001 → val_rmse=1.2096
 tried gens=100 pc=0.00

In [18]:
# START K=6 rich
outdir = run_strategy2_for_one_K(
    X_train=X_train, X_test=X_test, y_train=y_train, y_test=y_test,
    K=6,
    fs_name="rich",
    funcset=None,
    gen_grid=(50,100,200),
    parsimony_grid=(0.001,0.005,0.01),
    population_size=1000,
    out_root="artifacts",
    random_state=42
)
print("Started run ->", outdir)

[K=6 | rich] Groups: [['AveOccup'], ['Population'], ['MedInc'], ['HouseAge'], ['AveRooms', 'AveBedrms'], ['Latitude', 'Longitude']]
No function_set object found; proceeding with funcset as passed.
[K=6] Processing group 1/6 (cols [5]) ...
 tried gens=50 pc=0.001 → val_rmse=1.1551
 tried gens=50 pc=0.005 → val_rmse=1.2125
 tried gens=50 pc=0.01 → val_rmse=1.2125
 tried gens=100 pc=0.001 → val_rmse=1.1551
 tried gens=100 pc=0.005 → val_rmse=1.2125
 tried gens=100 pc=0.01 → val_rmse=1.2125
 tried gens=200 pc=0.001 → val_rmse=1.1468
 tried gens=200 pc=0.005 → val_rmse=1.1467
 tried gens=200 pc=0.01 → val_rmse=1.2125
Best SR: gens=200 pc=0.005 val_rmse=1.1467
Saved partial after group 1 to artifacts/strategy2_california_K6_rich_20251227_125334/partial_state.pkl
[K=6] Processing group 2/6 (cols [4]) ...
 tried gens=50 pc=0.001 → val_rmse=1.2096
 tried gens=50 pc=0.005 → val_rmse=1.2118
 tried gens=50 pc=0.01 → val_rmse=1.2118
 tried gens=100 pc=0.001 → val_rmse=1.2096
 tried gens=100 pc=0.00

Saving Results

In [20]:
# Cell: collect + persist Strategy 2 results (create sr_equations_summary.csv + summary_for_thesis.json)
import json, pickle, joblib, traceback, re, time
from pathlib import Path
import pandas as pd
import numpy as np
import inspect

ART = Path("artifacts")
OUTROOT = ART / f"strategy2_summary_{time.strftime('%Y%m%d_%H%M%S')}"
OUTROOT.mkdir(parents=True, exist_ok=True)

def try_extract_expr(est):
    """Try many strategies to convert an SR estimator into a readable expression string.
       Return dict with keys: expr (str|None), estimator_type (str), extra (dict)
    """
    info = {"expr": None, "estimator_type": type(est).__name__ if est is not None else None, "extra": {}}
    try:
        # If it's already a dict-like with metadata
        if isinstance(est, dict):
            # common keys used earlier
            for k in ("expr","expression","program","prog","str"):
                if k in est and est[k]:
                    info["expr"] = str(est[k])
                    return info
            # maybe stored as {'est': estimator, 'expr': ...}
            if "est" in est and est["est"] is not None:
                return try_extract_expr(est["est"])
        # If it's a tuple/list (est, cols, ...)
        if isinstance(est, (tuple, list)):
            # try first element
            return try_extract_expr(est[0])
        if est is None:
            return info

        # gplearn SymbolicRegressor: try _program or program or get_program
        for attr in ("_program", "program", "best_program"):
            prog = getattr(est, attr, None)
            if prog is not None:
                # Program object in gplearn prints nicely
                try:
                    s = str(prog)
                    if s and (len(s) < 10000):
                        info["expr"] = s
                        return info
                except Exception:
                    pass

        # Some wrappers may store ._program.export() or .export()
        try:
            if hasattr(est, "export"):
                try:
                    s = est.export()
                    if s:
                        info["expr"] = str(s)
                        return info
                except Exception:
                    pass
        except Exception:
            pass

        # Some implementations keep expression in attribute `symbolic_expression` or `expression`
        for attr in ("symbolic_expression","expression","expr","_expr","_expression"):
            val = getattr(est, attr, None)
            if val:
                info["expr"] = str(val)
                return info

        # Try __str__ of estimator (sometimes the Program string is embedded)
        try:
            s = str(est)
            # heuristics: string should contain math symbols or variable names like X0, X1, or operations
            if s and (re.search(r"[+\-*/^]|X\d+|sin|cos|log|sqrt", s) or len(s) < 2000):
                info["expr"] = s
                return info
        except Exception:
            pass

        # For sklearn Pipeline / custom wrapper, try to find a final step with symbolic program
        if hasattr(est, "steps"):
            for _, step in getattr(est, "steps"):
                r = try_extract_expr(step)
                if r["expr"]:
                    return r

        # For joblib-laden objects, show repr
        info["extra"]["repr"] = repr(est)[:2000]
    except Exception as e:
        info["extra"]["error"] = traceback.format_exc()[:2000]
    return info

# collect all strategy2 run folders
run_dirs = sorted([d for d in ART.iterdir() if d.is_dir() and d.name.startswith("strategy2_california")], key=lambda d: d.stat().st_mtime)
print(f"Found {len(run_dirs)} strategy2 runs under {ART}")

rows = []               # rows for sr_equations_summary.csv
run_level_rows = []     # rows for all_runs_summary.csv
detailed_sr_entries = []  # for JSON

for d in run_dirs:
    run_id = d.name
    meta = {}
    meta_path = d / "meta.json"
    if meta_path.exists():
        try:
            meta = json.loads(meta_path.read_text())
        except Exception:
            try:
                meta = json.loads((d / "meta.json").read_text(errors='ignore'))
            except:
                meta = {"meta_load_error": True}
    # read summary.csv if exists
    summary_csv = d / "summary.csv"
    run_summary_df = None
    if summary_csv.exists():
        try:
            run_summary_df = pd.read_csv(summary_csv)
        except Exception:
            try:
                run_summary_df = pd.read_csv(summary_csv, engine="python")
            except:
                run_summary_df = None

    # read sr_models.pkl or sr_models.joblib or sr_models.*
    srmodels = {}
    for fname in ("sr_models.pkl","sr_models.joblib","sr_models.pkl.joblib"):
        p = d / fname
        if p.exists():
            try:
                srmodels = joblib.load(str(p))
            except Exception:
                try:
                    with open(p,"rb") as f:
                        srmodels = pickle.load(f)
                except Exception:
                    srmodels = {"load_error_repr": repr(p)}
            break

    # fallback: inspect partial_state.pkl as it contains sr_models sometimes
    if (not srmodels) and (d/"partial_state.pkl").exists():
        try:
            partial = joblib.load(str(d/"partial_state.pkl"))
        except Exception:
            try:
                partial = pickle.loads((d/"partial_state.pkl").read_bytes())
            except Exception:
                partial = None
        if isinstance(partial, dict) and "sr_models" in partial:
            srmodels = partial.get("sr_models", {})

    # Normalize srmodels: if it's a list of (name,val), convert to dict
    if isinstance(srmodels, (list, tuple)):
        try:
            srmodels = {k:v for k,v in srmodels}
        except Exception:
            # try sequence of dicts with name inside
            tmp = {}
            for i,v in enumerate(srmodels):
                name = getattr(v, "name", f"SR_{i+1}")
                tmp[name] = v
            srmodels = tmp

    # If srmodels is empty dict still, note it
    if not isinstance(srmodels, dict):
        # maybe it's a wrapper object with attribute 'sr_models'
        if hasattr(srmodels, "sr_models"):
            try:
                srmodels = getattr(srmodels, "sr_models")
            except:
                pass

    # Build run-level row
    run_meta_row = {
        "run_id": run_id,
        "path": str(d),
        "meta": meta,
        "has_sr_models": bool(srmodels),
        "summary_csv_exists": summary_csv.exists()
    }
    # if summary df present, extract baseline/aug metrics if possible
    if run_summary_df is not None:
        try:
            # expect columns like Metric, Baseline, +SR
            # we will pivot to create baseline/aug numbers
            metrics = {}
            for idx,row in run_summary_df.iterrows():
                k = row.get("Metric") or row.get("metric") or f"m{idx}"
                metrics[k + "_Baseline"] = row.get("Baseline")
                metrics[k + "_Aug"] = row.get("+SR") if "+SR" in row else row.get("Aug") if "Aug" in row else row.get("Augmented")
            run_meta_row.update(metrics)
        except Exception:
            pass
    run_level_rows.append(run_meta_row)

    # iterate SRs
    if isinstance(srmodels, dict):
        for sr_name, bundle in srmodels.items():
            # normalize bundle to dict with 'est' and 'cols' if possible
            cols = None
            est = None
            val_rmse = None
            gens = None
            pc = None
            try:
                if isinstance(bundle, dict):
                    est = bundle.get("est") or bundle.get("estimator") or bundle.get("estimator_obj") or bundle.get("estimator_object")
                    cols = bundle.get("cols") or bundle.get("columns") or bundle.get("features")
                    val_rmse = bundle.get("val_rmse") or bundle.get("val_score") or bundle.get("rmse")
                    gens = bundle.get("gens") or bundle.get("generations")
                    pc = bundle.get("pc") or bundle.get("parsimony_coefficient") or bundle.get("parsimony")
                elif isinstance(bundle, (tuple, list)) and len(bundle) >= 1:
                    # (est, cols, ...)
                    est = bundle[0]
                    if len(bundle) > 1:
                        cols = bundle[1]
                else:
                    est = bundle
            except Exception:
                est = bundle

            # attempt to extract expression
            res = try_extract_expr(est)

            # if expr blank, try reading accompanying meta files named like SR_G*.meta etc
            if (not res["expr"]) and (d/"meta.json").exists():
                try:
                    m = json.loads((d/"meta.json").read_text())
                    # sometimes meta has sr entries
                    s = m.get("sr_models", {}) if isinstance(m, dict) else {}
                    if isinstance(s, dict) and sr_name in s:
                        cand = s[sr_name]
                        if isinstance(cand, dict) and cand.get("expr"):
                            res["expr"] = cand.get("expr")
                        elif isinstance(cand, str):
                            res["expr"] = cand
                except Exception:
                    pass

            row = {
                "run_id": run_id,
                "run_path": str(d),
                "K": meta.get("K"),
                "fs_name": meta.get("fs_name"),
                "sr_name": sr_name,
                "cols": cols,
                "expr": res.get("expr"),
                "estimator_type": res.get("estimator_type"),
                "val_rmse": val_rmse,
                "gens": gens,
                "parsimony_coefficient": pc
            }
            # include truncated repr for manual inspection
            try:
                row["estimator_repr"] = repr(est)[:2000]
            except Exception:
                row["estimator_repr"] = None

            rows.append(row)
            detailed_sr_entries.append(row)
    else:
        # no srmodels found
        rows.append({
            "run_id": run_id,
            "run_path": str(d),
            "K": meta.get("K"),
            "fs_name": meta.get("fs_name"),
            "sr_name": None,
            "cols": None,
            "expr": None,
            "estimator_type": None,
            "val_rmse": None,
            "gens": None,
            "parsimony_coefficient": None,
            "estimator_repr": None
        })

# Save sr_equations_summary.csv
df_sr = pd.DataFrame(rows)
csv_path = OUTROOT / "sr_equations_summary.csv"
df_sr.to_csv(csv_path, index=False)
print("Wrote:", csv_path)

# Save aggregated run-level CSV
df_runs = pd.DataFrame(run_level_rows)
runs_csv = OUTROOT / "all_runs_summary.csv"
df_runs.to_csv(runs_csv, index=False)
print("Wrote:", runs_csv)

# Save JSON summary_for_thesis.json
summary_json = {
    "generated_at": time.strftime("%Y-%m-%d %H:%M:%S"),
    "n_runs_found": len(run_dirs),
    "runs": run_level_rows,
    "sr_entries": detailed_sr_entries
}
json_path = OUTROOT / "summary_for_thesis.json"
json_path.write_text(json.dumps(summary_json, indent=2))
print("Wrote:", json_path)

# quick report
print("\nQuick report:")
print(" total SR rows:", len(df_sr))
print(" runs with sr_models:", df_sr['run_id'].nunique())
print(" sample SR rows (first 10):")
display(df_sr.head(10))

print("\nIf any 'expr' fields are null, open the corresponding run folder and inspect sr_models.pkl or partial_state.pkl manually.")
print("You can download the folder:", OUTROOT)

Found 16 strategy2 runs under artifacts
Wrote: artifacts/strategy2_summary_20251227_165116/sr_equations_summary.csv
Wrote: artifacts/strategy2_summary_20251227_165116/all_runs_summary.csv
Wrote: artifacts/strategy2_summary_20251227_165116/summary_for_thesis.json

Quick report:
 total SR rows: 39
 runs with sr_models: 9
 sample SR rows (first 10):


,run_id,run_path,K,fs_name,sr_name,cols,expr,estimator_type,val_rmse,gens,parsimony_coefficient,estimator_repr
0,strategy2_california_20251224_083012_K3_rich,artifacts/strategy2_california_20251224_083012...,NaN,None,SR_G1,"[MedInc, HouseAge, AveRooms, AveBedrms, Popula...","add(cos(sqrt(sqrt(sqrt(X2)))), add(cos(0.002),...",SymbolicRegressor,None,None,None,"SymbolicRegressor(function_set=('add', 'sub', ..."
1,strategy2_california_20251224_083012_K3_rich,artifacts/strategy2_california_20251224_083012...,NaN,None,SR_G2,[AveOccup],"sub(sqrt(X0), log(add(0.194, X0)))",SymbolicRegressor,None,None,None,"SymbolicRegressor(function_set=('add', 'sub', ..."
2,strategy2_california_20251224_083012_K3_rich,artifacts/strategy2_california_20251224_083012...,NaN,None,SR_G3,"[Latitude, Longitude]","add(add(cos(sin(X0)), sqrt(cos(add(add(add(sin...",SymbolicRegressor,None,None,None,"SymbolicRegressor(function_set=('add', 'sub', ..."
3,strategy2_california_K3_base_20251224_160434,artifacts/strategy2_california_K3_base_2025122...,3.0,None,SR_G1,"[MedInc, HouseAge, AveRooms, AveBedrms, Popula...","add(add(0.979, X0), 0.979)",SymbolicRegressor,None,None,None,"SymbolicRegressor(generations=50, parsimony_co..."
4,strategy2_california_K3_base_20251224_160434,artifacts/strategy2_california_K3_base_2025122...,3.0,None,SR_G2,[AveOccup],"div(0.994, sub(X0, -0.539))",SymbolicRegressor,None,None,None,"SymbolicRegressor(generations=200, parsimony_c..."
5,strategy2_california_K3_base_20251224_160434,artifacts/strategy2_california_K3_base_2025122...,3.0,None,SR_G3,"[Latitude, Longitude]","add(0.966, sub(0.873, add(X0, X1)))",SymbolicRegressor,None,None,None,"SymbolicRegressor(generations=50, parsimony_co..."
6,strategy2_california_K3_rich_20251224_165519,artifacts/strategy2_california_K3_rich_2025122...,3.0,rich,SR_G1,"[0, 1, 2, 3, 4]","add(cos(sqrt(sqrt(sqrt(X2)))), add(cos(0.002),...",SymbolicRegressor,None,None,None,"SymbolicRegressor(function_set=('add', 'sub', ..."
7,strategy2_california_K3_rich_20251224_165519,artifacts/strategy2_california_K3_rich_2025122...,3.0,rich,SR_G2,[5],"sub(sqrt(X0), log(add(0.194, X0)))",SymbolicRegressor,None,None,None,"SymbolicRegressor(function_set=('add', 'sub', ..."
8,strategy2_california_K3_rich_20251224_165519,artifacts/strategy2_california_K3_rich_2025122...,3.0,rich,SR_G3,"[6, 7]","add(add(cos(sin(X0)), sqrt(cos(add(add(add(sin...",SymbolicRegressor,None,None,None,"SymbolicRegressor(function_set=('add', 'sub', ..."
9,strategy2_california_K4_base_20251224_191521,artifacts/strategy2_california_K4_base_2025122...,4.0,base,SR_G1,[5],"div(0.994, sub(X0, -0.539))",SymbolicRegressor,None,None,None,"SymbolicRegressor(generations=200, parsimony_c..."



If any 'expr' fields are null, open the corresponding run folder and inspect sr_models.pkl or partial_state.pkl manually.
You can download the folder: artifacts/strategy2_summary_20251227_165116


# Strategy 1

In [9]:
# Cell 0 — sanity / environment
import sys, os, multiprocessing, psutil
print("Python:", sys.version)
print("Executable:", sys.executable)
print("CWD:", os.getcwd())
print("CPU count:", multiprocessing.cpu_count())
print("Total RAM (GiB):", psutil.virtual_memory().total/1024**3)

# check package versions we need
import importlib
def v(name):
    try:
        m = importlib.import_module(name)
        return getattr(m, "__version__", "unknown")
    except Exception as e:
        return f"NOT INSTALLED ({type(e).__name__})"
for pkg in ("numpy","pandas","sklearn","gplearn","xgboost","joblib"):
    print(f"{pkg:10s} -> {v(pkg)}")

# set thread envs (adapt number to what you reserved on SOL)
CPU = 12   # ← adjust if you reserved a different number
for k in ("OMP_NUM_THREADS","OPENBLAS_NUM_THREADS","MKL_NUM_THREADS","NUMEXPR_NUM_THREADS"):
    os.environ[k] = str(CPU)
print("Thread envs:", {k:os.environ.get(k) for k in ('OMP_NUM_THREADS','OPENBLAS_NUM_THREADS','MKL_NUM_THREADS')})

Python: 3.12.9 | packaged by conda-forge | (main, Mar  4 2025, 22:48:41) [GCC 13.3.0]
Executable: /packages/apps/jupyter/2025-03-24/bin/python3.12
CWD: /home/rshayan1
CPU count: 128
Total RAM (GiB): 503.47483825683594
numpy      -> 1.26.4
pandas     -> 2.3.3
sklearn    -> 1.2.2
gplearn    -> 0.4.2
xgboost    -> 2.1.1
joblib     -> 1.5.3
Thread envs: {'OMP_NUM_THREADS': '12', 'OPENBLAS_NUM_THREADS': '12', 'MKL_NUM_THREADS': '12'}


In [10]:
# Cell 1 — Strategy-1 functions (copied from your original S1 cells; run once)
import os, json, time, uuid, math, joblib, numpy as np, pandas as pd
from pathlib import Path
from itertools import combinations
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error
from sklearn.linear_model import LinearRegression, Ridge
from xgboost import XGBRegressor
from gplearn.genetic import SymbolicRegressor
rng = np.random.default_rng(42)

ART_DIR = Path("artifacts") / "strategy1"
ART_DIR.mkdir(parents=True, exist_ok=True)

# ---- dataset loader (same as you had) ----
def load_cal_housing(test_size=0.2, seed=42):
    from sklearn.datasets import fetch_california_housing
    data = fetch_california_housing(as_frame=True)
    X, y = data.data.copy(), data.target.copy()
    # keep raw names to report groups
    names = list(X.columns)
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=test_size, random_state=seed)
    ss = StandardScaler().fit(X_train.values)
    X_train_s = pd.DataFrame(ss.transform(X_train.values), columns=names, index=X_train.index)
    X_test_s  = pd.DataFrame(ss.transform(X_test.values),  columns=names, index=X_test.index)
    return (X_train_s, X_test_s, y_train.values, y_test.values, names)

# ---- quick metrics ----
def rmse(y_true, y_pred): return float(np.sqrt(mean_squared_error(y_true, y_pred)))

def eval_baselines(Xtr, Xte, ytr, yte):
    out = {}
    for name, model in {
        "LR": LinearRegression(),
        "Ridge": Ridge(alpha=1.0, random_state=42),
        "XGB": XGBRegressor(
            n_estimators=300, max_depth=4, learning_rate=0.05,
            subsample=0.9, colsample_bytree=0.9, random_state=42, n_jobs=-1,
        ),
    }.items():
        model.fit(Xtr, ytr)
        out[f"{name}_RMSE"] = rmse(yte, model.predict(Xte))
    return out

# ---- SR helper ----
def fit_sr_feature(Xtr_group, ytr, gens=50, pop=800, pc=0.005, seed=42, fset=('add','sub','mul','div')):
    est = SymbolicRegressor(
        function_set=fset, generations=gens, population_size=pop,
        parsimony_coefficient=pc, random_state=seed, n_jobs=1, verbose=0
    )
    est.fit(Xtr_group, ytr.ravel())
    return est

def add_sr_features(X, sr_models, groups):
    X_aug = X.copy()
    for gi, model in sr_models.items():
        name = f"SR_G{gi+1}"
        X_aug[name] = model.predict(X.iloc[:, groups[gi]])
    return X_aug

# ---- grouping functions (farthest-subset) ----
from sklearn.metrics import pairwise_distances
def corr_dissimilarity(X):  # 1 - |corr|
    C = np.corrcoef(X, rowvar=False)
    D = 1.0 - np.abs(C)
    np.fill_diagonal(D, 0.0)
    return D

def farthest_subset_groups(X: pd.DataFrame, K: int, seed: int = 42):
    rng = np.random.default_rng(seed)
    n, d = X.shape
    D = corr_dissimilarity(X.values)
    feats = list(range(d))
    anchors = []
    scores = D.mean(axis=1)
    anchors.append(int(np.argmax(scores)))
    while len(anchors) < K:
        rem = [j for j in feats if j not in anchors]
        min_d = [min(D[j, a] for a in anchors) for j in rem]
        anchors.append(rem[int(np.argmax(min_d))])
    groups = [[a] for a in anchors]
    remaining = [j for j in feats if j not in anchors]
    for j in remaining:
        group_ds = [D[j, grp].mean() for grp in groups]
        gid = int(np.argmax(group_ds))
        groups[gid].append(j)
    lens = [len(g) for g in groups]
    while max(lens) - min(lens) > 2:
        g_big = int(np.argmax(lens)); g_small = int(np.argmin(lens))
        cand = groups[g_big][:]
        losses = [D[i, groups[g_big]].mean() for i in cand]
        take = cand[int(np.argmin(losses))]
        groups[g_big].remove(take); groups[g_small].append(take)
        lens = [len(g) for g in groups]
    return groups

print("Strategy-1 functions loaded. ART_DIR:", ART_DIR)

Strategy-1 functions loaded. ART_DIR: artifacts/strategy1


In [11]:
# Cell 2 — run Strategy-1 and save artifacts (copy+run)
import os, json, time
from pathlib import Path
import joblib, pandas as pd, numpy as np
from datetime import datetime

# CONFIG — change these if you want fewer gens/pops or different K
FORCE = False
K_list = [3,4,5,6]            # runs to execute (change if you only want some)
function_sets = {"base": ('add','sub','mul','div'),
                 "rich": ('add','sub','mul','div','sin','cos')}
gens_grid = [50]              # keep small for speed; expand later if desired
pop_grid = [800]
pc_grid = [0.005]
seed = 42

ART_ROOT = Path("artifacts") / "strategy1"
ART_ROOT.mkdir(parents=True, exist_ok=True)

# helper atomic write
def atomic_write(path: Path, obj):
    path.parent.mkdir(parents=True, exist_ok=True)
    tmp = path.with_suffix(path.suffix + ".tmp")
    with open(tmp, "w") as f:
        json.dump(obj, f, indent=2)
    os.replace(tmp, path)

# main loop
Xtr, Xte, ytr, yte, NAMES = load_cal_housing()
rows = []
for K in K_list:
    groups = farthest_subset_groups(Xtr, K, seed=seed)
    for fs_name, fset in function_sets.items():
        run_id = f"cal_housing_S1_K{K}_{fs_name}"
        run_dir = ART_ROOT / run_id
        models_dir = run_dir / "models"
        run_dir.mkdir(parents=True, exist_ok=True)
        models_dir.mkdir(parents=True, exist_ok=True)
        meta_path = run_dir / "meta.json"
        sum_path  = run_dir / "summary.json"
        eq_path   = run_dir / "equations.csv"
        sr_models_pkl = run_dir / "sr_models.pkl"

        if sum_path.exists() and (not FORCE):
            print(f"[skip] {run_id} summary exists → skipping (set FORCE=True to overwrite)")
            # load summary for leaderboard row
            s = json.load(open(sum_path))
            rows.append({"run_id":run_id, "K":K, "fs":fs_name, "mean_dRMSE": s["dRMSE"]["mean_dRMSE"], "summary_path": str(sum_path)})
            continue

        # meta
        if not meta_path.exists():
            atomic_write(meta_path, {
                "strategy":"S1_farthest_subsets",
                "dataset":"cal_housing",
                "run_id":run_id,
                "K":K, "function_set":fs_name,
                "groups":[[NAMES[j] for j in g] for g in groups],
                "created_at": datetime.now().isoformat()
            })

        # baseline
        base = eval_baselines(Xtr, Xte, ytr, yte)

        # per-group SR training (checkpoint per group)
        sr_status = {"trained": [], "best": {}}
        sr_models = {}
        for gi, g in enumerate(groups):
            model_file = models_dir / f"sr_g{gi+1}.joblib"
            if model_file.exists() and (gi in sr_status["trained"]) and (not FORCE):
                sr_models[gi] = joblib.load(model_file)
                continue

            # train grid and pick best by val RMSE
            best = (math.inf, None, None)
            Xg = Xtr.iloc[:, g]
            for gens in gens_grid:
                for pop in pop_grid:
                    for pc in pc_grid:
                        est = fit_sr_feature(Xg, ytr, gens=gens, pop=pop, pc=pc, seed=seed, fset=fset)
                        Xg_tr, Xg_val, y_tr, y_val = train_test_split(Xg, ytr, test_size=0.15, random_state=7)
                        v = rmse(y_val, est.predict(Xg_val))
                        if v < best[0]:
                            best = (v, est, {"gens":gens, "pop":pop, "pc":pc})
            # save model & metadata
            joblib.dump(best[1], model_file)
            sr_status["trained"] = sorted(list(set(sr_status["trained"] + [gi])))
            sr_status["best"][f"G{gi+1}"] = {"val_rmse": best[0], **best[2]}
            atomic_write(run_dir / "sr_status.json", sr_status)
            sr_models[gi] = best[1]
            print(f"[{run_id}] trained SR_G{gi+1} (val RMSE={best[0]:.4f})")

        # build SR features
        Xtr_aug = add_sr_features(Xtr, sr_models, groups)
        Xte_aug = add_sr_features(Xte, sr_models, groups)
        sr_cols = [c for c in Xtr_aug.columns if c.startswith("SR_G")]

        # keep selection
        def keep_mask_from_val_local(sr_cols, Xtr_aug, ytr, q=0.75):
            vals=[]
            for c in sr_cols:
                m = LinearRegression().fit(Xtr_aug[[c]], ytr)
                vals.append(rmse(ytr, m.predict(Xtr_aug[[c]])))
            thr = np.quantile(vals,q)
            kept=[c for c,v in zip(sr_cols,vals) if v <= thr]
            return kept, dict(zip(sr_cols,vals)), float(thr)
        keep_cols, val_table, thr = keep_mask_from_val_local(sr_cols, Xtr_aug, ytr, q=0.75)
        atomic_write(run_dir / "kept.json", {"kept": keep_cols, "val_rmse_per_col": val_table, "threshold": thr})

        Xtr_final = Xtr.join(Xtr_aug[keep_cols])
        Xte_final = Xte.join(Xte_aug[keep_cols])

        # evaluate + finalize summary
        aug = eval_baselines(Xtr_final, Xte_final, ytr, yte)
        dRMSE = {
            "LR_dRMSE": aug["LR_RMSE"] - base["LR_RMSE"],
            "Ridge_dRMSE": aug["Ridge_RMSE"] - base["Ridge_RMSE"],
            "XGB_dRMSE": aug["XGB_RMSE"] - base["XGB_RMSE"],
        }
        dRMSE["mean_dRMSE"] = float(np.mean(list(dRMSE.values())))

        summary = {
            "strategy": "S1_farthest_subsets",
            "dataset": "cal_housing",
            "run_id": run_id,
            "K": K,
            "function_set": fs_name,
            "groups": [[NAMES[j] for j in g] for g in groups],
            "kept_features": keep_cols,
            "baseline": base,
            "augmented": aug,
            "dRMSE": dRMSE,
            "checkpoints": {
                "meta": str(meta_path),
                "sr_status": str(run_dir / "sr_status.json"),
                "kept": str(run_dir / "kept.json")
            },
            "completed_at": datetime.now().isoformat()
        }
        atomic_write(sum_path, summary)

        # persist sr_models.pkl (all estimates)
        with open(sr_models_pkl, "wb") as f:
            joblib.dump(sr_models, f)

        # persist augmented features as parquet
        try:
            Xtr_final.to_parquet(run_dir / "X_train_aug.parquet")
            Xte_final.to_parquet(run_dir / "X_test_aug.parquet")
        except Exception:
            joblib.dump(Xtr_final, run_dir / "X_train_aug.pkl")
            joblib.dump(Xte_final, run_dir / "X_test_aug.pkl")

        # save small CSV summary line for leaderboard convenience
        df_line = pd.DataFrame([{
            "run_id": run_id, "K": K, "fs": fs_name,
            "LR_dRMSE": dRMSE["LR_dRMSE"],
            "Ridge_dRMSE": dRMSE["Ridge_dRMSE"],
            "XGB_dRMSE": dRMSE["XGB_dRMSE"],
            "mean_dRMSE": dRMSE["mean_dRMSE"],
            "kept": ",".join(keep_cols),
            "summary_path": str(sum_path)
        }])
        df_line.to_csv(run_dir / "summary.csv", index=False)

        # extract equations if possible
        equations = {}
        for gi, model in sr_models.items():
            try:
                if hasattr(model, "program"):
                    equations[f"SR_G{gi+1}"] = str(model.program)
                else:
                    equations[f"SR_G{gi+1}"] = repr(model)
            except Exception as e:
                equations[f"SR_G{gi+1}"] = f"<error: {e}>"
        # save equations csv/json
        pd.DataFrame([{"sr_name":k, "expr":v, "run_id": run_id} for k,v in equations.items()]).to_csv(eq_path, index=False)
        atomic_write(run_dir / "meta.json", {"K":K, "fs_name":fs_name, "run_id":run_id, "created_at":datetime.now().isoformat()})

        print(f"[DONE] {run_id}  → mean Δ {dRMSE['mean_dRMSE']:+.4f}")

        rows.append({"run_id":run_id, "K":K, "fs":fs_name, "mean_dRMSE": dRMSE["mean_dRMSE"], "summary_path": str(sum_path)})

# Save leaderboard for S1
leader = pd.DataFrame(rows).sort_values("mean_dRMSE")
leader_path = ART_ROOT / "strategy1_leaderboard.csv"
leader.to_csv(leader_path, index=False)
print("Saved leaderboard:", leader_path)

[cal_housing_S1_K3_base] trained SR_G1 (val RMSE=1.1798)
[cal_housing_S1_K3_base] trained SR_G2 (val RMSE=0.8814)
[cal_housing_S1_K3_base] trained SR_G3 (val RMSE=1.1000)
[DONE] cal_housing_S1_K3_base  → mean Δ +0.0006
[cal_housing_S1_K3_rich] trained SR_G1 (val RMSE=1.1494)
[cal_housing_S1_K3_rich] trained SR_G2 (val RMSE=0.8779)
[cal_housing_S1_K3_rich] trained SR_G3 (val RMSE=1.0970)
[DONE] cal_housing_S1_K3_rich  → mean Δ +0.0006
[cal_housing_S1_K4_base] trained SR_G1 (val RMSE=1.1798)
[cal_housing_S1_K4_base] trained SR_G2 (val RMSE=1.1845)
[cal_housing_S1_K4_base] trained SR_G3 (val RMSE=1.1842)
[cal_housing_S1_K4_base] trained SR_G4 (val RMSE=0.8770)
[DONE] cal_housing_S1_K4_base  → mean Δ +0.0005
[cal_housing_S1_K4_rich] trained SR_G1 (val RMSE=1.1494)
[cal_housing_S1_K4_rich] trained SR_G2 (val RMSE=1.1688)
[cal_housing_S1_K4_rich] trained SR_G3 (val RMSE=1.1842)
[cal_housing_S1_K4_rich] trained SR_G4 (val RMSE=0.8873)
[DONE] cal_housing_S1_K4_rich  → mean Δ +0.0003
[cal_housi

In [12]:
# Cell 3 — collate S1 runs and write unified summary_for_thesis + equations table
import json, pandas as pd
from pathlib import Path
from datetime import datetime

ART_ROOT = Path("artifacts") / "strategy1"
OUT_ROOT = Path("artifacts") / f"strategy1_summary_{datetime.now().strftime('%Y%m%d_%H%M%S')}"
OUT_ROOT.mkdir(parents=True, exist_ok=True)

summaries = []
equations_all = []
for d in sorted(ART_ROOT.iterdir()):
    if not d.is_dir(): continue
    s = d / "summary.json"
    eq = d / "equations.csv"
    if s.exists():
        sj = json.load(open(s))
        summaries.append(sj)
    if eq.exists():
        eqdf = pd.read_csv(eq)
        equations_all.append(eqdf)

# write combined summary csv
if summaries:
    rows = []
    for sj in summaries:
        rows.append({
            "run_id": sj["run_id"],
            "K": sj["K"],
            "fs": sj["function_set"],
            "mean_dRMSE": sj["dRMSE"]["mean_dRMSE"],
            "LR_dRMSE": sj["dRMSE"]["LR_dRMSE"],
            "Ridge_dRMSE": sj["dRMSE"]["Ridge_dRMSE"],
            "XGB_dRMSE": sj["dRMSE"]["XGB_dRMSE"],
            "kept": ",".join(sj.get("kept_features", []))
        })
    df_all = pd.DataFrame(rows).sort_values("mean_dRMSE")
    df_all.to_csv(OUT_ROOT / "all_runs_summary.csv", index=False)
    # summary_for_thesis.json
    with open(OUT_ROOT / "summary_for_thesis.json","w") as f:
        json.dump({"generated_at": datetime.now().isoformat(), "runs": summaries}, f, indent=2)
    print("Wrote combined summary and JSON to:", OUT_ROOT)

# collate equations
if equations_all:
    eq_all = pd.concat(equations_all, ignore_index=True)
    eq_all.to_csv(OUT_ROOT / "sr_equations_summary.csv", index=False)
    print("Wrote combined equations table:", OUT_ROOT / "sr_equations_summary.csv")

print("Done. OUT_ROOT:", OUT_ROOT)

Wrote combined summary and JSON to: artifacts/strategy1_summary_20251228_152535
Wrote combined equations table: artifacts/strategy1_summary_20251228_152535/sr_equations_summary.csv
Done. OUT_ROOT: artifacts/strategy1_summary_20251228_152535
